<a href="https://colab.research.google.com/github/nerfstudio-project/nerfstudio/blob/alex%2Fcolab-logo-fix/colab/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
    <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://docs.nerf.studio/_images/logo-dark.png">
    <source media="(prefers-color-scheme: light)" srcset="https://docs.nerf.studio/_images/logo.png">
    <img alt="nerfstudio" src="https://docs.nerf.studio/_images/logo.png" width="400">
    </picture>
</p>


# Nerfstudio: A collaboration friendly studio for NeRFs


![GitHub stars](https://img.shields.io/github/stars/nerfstudio-project/nerfstudio?color=gold&style=social)

This colab shows how to train and view NeRFs from Nerfstudio both on pre-made datasets or from your own videos/images.

\\

Credit to [NeX](https://nex-mpi.github.io/) for Google Colab format.

## Frequently Asked Questions

*  **Downloading custom data is stalling (no output):**
    * This is a bug in Colab. The data is processing, but may take a while to complete. You will know processing completed if `data/nerfstudio/custom_data/transforms.json` exists. Terminating the cell early will result in not being able to train.
*  **Processing custom data is taking a long time:**
    * The time it takes to process data depends on the number of images and its resolution. If processing is taking too long, try lowering the resolution of your custom data.
*  **Error: Data processing did not complete:**
    * This means that the data processing script did not fully complete. This could be because there were not enough images, or that the images were of low quality. We recommend images with little to no motion blur and lots of visual overlap of the scene to increase the chances of successful processing.
*   **Training is not showing progress**:
    * The lack of output is a bug in Colab. You can see the training progress from the viewer.
* **Viewer Quality is bad / Low resolution**:
    * This may be because more GPU is being used on training that rendering the viewer. Try pausing training or decreasing training utilization.
* **WARNING: Running pip as the 'root' user...:**:
    * This and other pip warnings or errors can be safely ignored.
* **Other problems?**
    * Feel free to create an issue on our [GitHub repo](https://github.com/nerfstudio-project/nerfstudio).


In [ ]:
# @markdown <h1>Install Nerfstudio and Dependencies (~8 min)</h1>

%cd /content/
!pip install --upgrade pip
# Colab's preinstalled torch is used; splatfacto does not need TinyCuda

# Installing COLMAP
%cd /content/
!apt-get install colmap

# Install nerfstudio
%cd /content/
!pip install nerfstudio==1.1.5

In [ ]:
# @markdown <h1> Downloading and Processing Data</h1>
# @markdown <h3>Upload colab_input.zip</h3>
import glob
import json
import os

import numpy as np
from google.colab import files
from IPython.core.display import HTML, display

%cd /content/
uploaded = files.upload()
input_zipfile = list(uploaded.keys())[0].replace(" ", "\\ ").replace("(", "\\(").replace(")", "\\)")
!unzip -o -q $input_zipfile -d /content/upload
!mkdir -p /content/data/nerfstudio
!rm -rf /content/data/nerfstudio/custom_data && mv /content/upload/data /content/data/nerfstudio/custom_data
scene = "custom_data"

print("Data Processing Succeeded!")

In [ ]:
# @markdown <h1>Start Training</h1>

%cd /content
if os.path.exists(f"data/nerfstudio/{scene}/transforms.json"):
    # compile gsplat's CUDA kernels once, quietly (its spinner floods the cell); one job per ~4 GB of RAM, so 1 on a 12 GB runtime
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    os.environ["MAX_JOBS"] = str(min(os.cpu_count(), int(ram_gb // 4)) if ram_gb > 24 else 1)
    print("gsplat compile with MAX_JOBS =", os.environ["MAX_JOBS"])
    !python -c "import gsplat.cuda._backend" > gsplat_build.log 2>&1 || tail -n 50 gsplat_build.log
    !ns-train splatfacto --max-num-iterations 15000 --viewer.quit-on-train-completion True nerfstudio-data --data data/nerfstudio/{scene} --downscale-factor 2
    if not glob.glob("outputs/unnamed/splatfacto/*/nerfstudio_models/*.ckpt"):
        raise RuntimeError("Training did not finish, re-run Start Training")
else:
    from IPython.core.display import HTML, display

    display(HTML('<h3 style="color:red">Error: Data processing did not complete</h3>'))
    display(HTML("<h3>Please re-run `Downloading and Processing Data`, or view the FAQ for more info.</h3>"))

In [ ]:
# @markdown <h1>Render Images</h1>

base_dir = "/content/outputs/unnamed/splatfacto/"
checkpoints = sorted(glob.glob(base_dir + "*/nerfstudio_models/*.ckpt"))
if not checkpoints:
    raise RuntimeError("Training did not finish, re-run Start Training")
training_run_dir = os.path.dirname(os.path.dirname(checkpoints[-1]))
config_filename = training_run_dir + "/config.yml"

# camera path is in transforms.json coordinates; move it into nerfstudio's normalized frame
dataparser_transforms = json.load(open(training_run_dir + "/dataparser_transforms.json"))
transform = np.vstack([np.array(dataparser_transforms["transform"]), [0, 0, 0, 1]])
camera_path = json.load(open("/content/upload/low_orbit_path.json"))
for camera in camera_path["camera_path"]:
    c2w = transform @ np.array(camera["camera_to_world"]).reshape(4, 4)
    c2w[:3, 3] *= dataparser_transforms["scale"]
    camera["camera_to_world"] = c2w.flatten().tolist()
camera_path_filename = training_run_dir + "/low_orbit_path.json"
json.dump(camera_path, open(camera_path_filename, "w"))

%cd /content/
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render camera-path --load-config $config_filename --camera-path-filename $camera_path_filename --output-format images --output-path renders/low_orbit
rendered = sorted(glob.glob("renders/low_orbit/0*.jpg"))
if _exit_code != 0 or len(rendered) != len(camera_path["camera_path"]):
    raise RuntimeError(f"ns-render failed (exit code {_exit_code}, {len(rendered)} images), re-run Render Images")
for i, camera in enumerate(camera_path["camera_path"]):
    os.rename(f"renders/low_orbit/{i:05d}.jpg", f"renders/low_orbit/{camera['name']}.jpg")

In [ ]:
# @markdown <h1>Export Splat</h1>

%cd /content/
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-eval --load-config $config_filename --output-path exports/splat/eval.json
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat --load-config $config_filename --output-dir exports/splat
!cp $config_filename $training_run_dir/dataparser_transforms.json exports/splat/
!zip -q -r renders.zip renders/low_orbit && zip -q -r splat_export.zip exports/splat
files.download("renders.zip")
files.download("splat_export.zip")